# SMS Spam - Text Cleaning And Model Comparison

Notebook này tập trung vào 1 dataset, chạy nhiều mô hình để so sánh và ghi lại kết quả thực nghiệm.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.svm import SVC

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
df = pd.read_csv(DATA_DIR / "sms_spam.csv")
df["label"] = df["label"].map({"ham": 0, "spam": 1})
display(df.head())

X_train, X_test, y_train, y_test = train_test_split(df["message"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)


In [ ]:
vectorizers = {
    "tfidf_unigram": TfidfVectorizer(max_features=5000, stop_words="english"),
    "tfidf_bigram": TfidfVectorizer(max_features=8000, ngram_range=(1, 2), stop_words="english"),
}

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVCProxy": SVC(kernel="linear"),
}

results = []
for vec_name, vectorizer in vectorizers.items():
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    for model_name, model in models.items():
        model.fit(X_train_vec, y_train)
        preds = model.predict(X_test_vec)
        results.append({
            "setup": f"{vec_name}+{model_name}",
            "accuracy": accuracy_score(y_test, preds),
            "f1_weighted": f1_score(y_test, preds, average="weighted"),
        })

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
display(results_df)
sns.barplot(data=results_df, x="accuracy", y="setup", palette="Oranges_r")
plt.title("SMS Spam Comparison")
plt.show()
